Goals:

- Create final objects for both voxel-level as well as cell-level data

In [1]:
from pathlib import Path
import os 

import pandas as pd
import numpy as np
import zarr
from zarr.core.dtype import VariableLengthUTF8
import scanpy as sc 
import scipy.sparse as sp

from dotenv import dotenv_values

from hm3d_utils.spatial import BoundingBox3D
from hm3d_utils.utils import write_clean_h5ad
from project_utils import FileTracker

In [ ]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

SPATIALDIR = DATADIR / "7_spatial"
OUTDIR = DATADIR / "9_objects" / "spatial"
os.makedirs(OUTDIR, exist_ok=True)

VOXEL_IN_FP = SPATIALDIR / 'sp_voxels.zarr'
SC_ST_IN_FP = SPATIALDIR / 'SP_concat.h5ad'

VOXEL_OUT_FP = OUTDIR / 'hypomap3d_spatial_voxel_level.zarr'
SC_ST_OUT_FP = OUTDIR / 'hypomap3d_spatial_cell_level.h5ad'

FLAGS_FP = SPATIALDIR / "analyses" / "spatial_flags.csv"

# adata for annotation"
SC_FP = DATADIR / "9_objects"/ "singlecell" / "hypomap3d_singlecell_control.h5ad" 

tracker = FileTracker(log_file=DATADIR/'tracked_files.toml')

### Let's build mappers for cell type labels 

In [ ]:
def build_id_mapping(n_df, id_pattern, target_pattern, levels=range(1, 6),):
    mapping = {}

    # --- 1. from wide dataframe ---
    for i in levels:
        id_col = id_pattern.format(i=i)
        target_col = target_pattern.format(i=i)

        if id_col in n_df.columns and target_col in n_df.columns:
            mapping.update(
                n_df[[id_col, target_col]]
                .dropna()
                .drop_duplicates()
                .set_index(id_col)[target_col]
                .to_dict()
            )

    return mapping

In [4]:
adata_sc = sc.read_h5ad(SC_FP, backed="r")

abclike_mapper = build_id_mapping(adata_sc.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_abc_like")

descr_mapper = build_id_mapping(adata_sc.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_descr") 

del adata_sc

### Let's load the flags 

In [5]:
flags_df = pd.read_csv(FLAGS_FP) 
flags_df.warning = flags_df.warning.fillna("None") 
flags_df.columns = ["ids", "flag_category"]
flags_df = flags_df.set_index("ids")
flag_mapper = flags_df.to_dict()["flag_category"]

## Let's process the voxel level object

- restructure
- add bounding box information
- update cluster labels
- add flags

In [6]:
voxel_old = zarr.open(VOXEL_IN_FP)
voxel_old.tree()

/
├── L1_mass (6, 168, 144, 107) float32
├── L1_prob (6, 168, 144, 107) float32
├── L1_rel_mass (6, 168, 144, 107) float32
├── L2_mass (82, 168, 144, 107) float32
├── L2_prob (82, 168, 144, 107) float32
├── L2_rel_mass (82, 168, 144, 107) float32
├── L3_mass (274, 168, 144, 107) float32
├── L3_prob (274, 168, 144, 107) float32
├── L3_rel_mass (274, 168, 144, 107) float32
├── L4_mass (716, 168, 144, 107) float32
├── L4_prob (716, 168, 144, 107) float32
├── L4_rel_mass (716, 168, 144, 107) float32
├── L5_mass (932, 168, 144, 107) float32
├── L5_prob (932, 168, 144, 107) float32
├── L5_rel_mass (932, 168, 144, 107) float32
├── counts (168, 144, 107) float32
├── evidence (168, 144, 107) float32
├── latent_embedding (45, 168, 144, 107) float32
├── region_labels (168, 144, 107) uint32
└── region_masks (176, 168, 144, 107) bool

In [7]:
voxel_new = zarr.open_group(VOXEL_OUT_FP, mode="w")

### add bounding box info

In [8]:
bbox_mm = BoundingBox3D.load(SPATIALDIR / "bounding_box_ccf_mm.json")
bbox_25um = BoundingBox3D.load(SPATIALDIR / 'bounding_box_voxels_25um.json')
bbox_mm, bbox_25um

(Bounding Box Ranges (CCFv3 mm):
  x: 4.375<->8.55
  y: 3.95<->7.525
  z: 3.05<->5.7,
 Bounding Box Ranges (25um Voxels):
  x: 175.0<->342.0
  y: 158.0<->301.0
  z: 122.0<->228.0)

In [ ]:
nx, ny, nz = voxel_old["L1_mass"].shape[1:]

x_mm_min = bbox_mm.min[0]
x_mm_max = bbox_mm.max[0]
y_mm_min = bbox_mm.min[1]
y_mm_max = bbox_mm.max[1]
z_mm_min = bbox_mm.min[2]
z_mm_max = bbox_mm.max[2]

x_um_min = bbox_25um.min[0]
x_um_max = bbox_25um.max[0]
y_um_min = bbox_25um.min[1]
y_um_max = bbox_25um.max[1]
z_um_min = bbox_25um.min[2]
z_um_max = bbox_25um.max[2]

In [10]:
bbox_grp = voxel_new.create_group("bbox", overwrite=True) 
bbox_grp.create_array(
    "shape",
    data=np.array([nx, ny, nz], dtype=np.float32),
    overwrite=True,
)

ccf_grp = bbox_grp.create_group("ccf_mm", overwrite=True)
ccf_grp.create_array(
    "min",
    data=np.array([x_mm_min, y_mm_min, z_mm_min], dtype=np.float32),
    overwrite=True,
)
ccf_grp.create_array(
    "max",
    data=np.array([x_mm_max, y_mm_max, z_mm_max], dtype=np.float32),
    overwrite=True,
)

vx_grp = bbox_grp.create_group("voxel_25_um", overwrite=True)
vx_grp.create_array(
    "min",
    data=np.array([x_um_min, y_um_min, z_um_min], dtype=np.float32),
    overwrite=True,
)
vx_grp.create_array(
    "max",
    data=np.array([x_um_max, y_um_max, z_um_max], dtype=np.float32),
    overwrite=True,
)

<Array file:///REDACTED-path/HypoMap3D_spatial_voxel_level.zarr/bbox/voxel_25_um/max shape=(3,) dtype=float32>

In [11]:
voxel_new.tree()

/
└── bbox
    ├── ccf_mm
    │   ├── max (3,) float32
    │   └── min (3,) float32
    ├── shape (3,) float32
    └── voxel_25_um
        ├── max (3,) float32
        └── min (3,) float32

### add level-wise data

In [ ]:
def copy_array(src_group, dst_group, src_key, dst_key=None, overwrite=True):
    if dst_key is None:
        dst_key = src_key

    src = src_group[src_key]

    dst = dst_group.create_array(
        dst_key,
        shape=src.shape,
        dtype=src.dtype,
        chunks=src.chunks,
        overwrite=overwrite,
    )

    dst[:] = src[:]
    return dst

In [13]:

for i in range(1, 6):
    grp_key = f"L{i}"

    # destination group
    lvl_grp = voxel_new.create_group(grp_key, overwrite=True)

    # source group references
    src_mass = voxel_old[f"{grp_key}_mass"]
    ids = src_mass.attrs["ids"]
    n_features = len(ids)

    # -------------------------
    # COPY VOLUMETRIC ARRAYS
    # -------------------------
    for arr_key in ["prob", "mass", "rel_mass"]:
        copy_array(
            voxel_old,
            lvl_grp,
            src_key=f"{grp_key}_{arr_key}",
            dst_key=arr_key,
        )

    # -------------------------
    # FLAGS / LABELS / ids
    # -------------------------
    lvl_grp.create_array(
        "flags",
        shape=(n_features,),
        dtype=VariableLengthUTF8(),
        overwrite=True,
    )
    lvl_grp["flags"][:] = [flag_mapper[i] for i in ids]

    lvl_grp.create_array(
        "descr",
        shape=(n_features,),
        dtype=VariableLengthUTF8(),
        overwrite=True,
    )
    lvl_grp["descr"][:] = [descr_mapper[i] for i in ids]

    lvl_grp.create_array(
        "abc-like",
        shape=(n_features,),
        dtype=VariableLengthUTF8(),
        overwrite=True,
    )
    lvl_grp["abc-like"][:] = [abclike_mapper[i] for i in ids]

    lvl_grp.create_array(
        "ids",
        shape=(n_features,),
        dtype=VariableLengthUTF8(),
        overwrite=True,
    )
    lvl_grp["ids"][:] = ids

### add rest

In [14]:
for k in ["counts", "evidence", "latent_embedding"]:
    copy_array(voxel_old, voxel_new, k)

In [15]:
region_old = voxel_old
region_new = voxel_new.create_group("region", overwrite=True)

for k in ["region_labels", "region_masks"]:
    copy_array(region_old, region_new, k)

mask_attrs = region_old["region_masks"].attrs

ids = np.array(mask_attrs["ids"], dtype=np.int32)

region_new.create_array(
    "ids",
    data=ids,
    overwrite=True,
)

acronyms = mask_attrs["acronyms"]

region_new.create_array(
    "acronyms",
    shape=(len(acronyms),),
    dtype=VariableLengthUTF8(),
    overwrite=True,
)

region_new["acronyms"][:] = acronyms

In [ ]:
voxel_new.tree()

/
├── L1
│   ├── abc-like (6,) StringDType()
│   ├── descr (6,) StringDType()
│   ├── flags (6,) StringDType()
│   ├── ids (6,) StringDType()
│   ├── mass (6, 168, 144, 107) float32
│   ├── prob (6, 168, 144, 107) float32
│   └── rel_mass (6, 168, 144, 107) float32
├── L2
│   ├── abc-like (82,) StringDType()
│   ├── descr (82,) StringDType()
│   ├── flags (82,) StringDType()
│   ├── ids (82,) StringDType()
│   ├── mass (82, 168, 144, 107) float32
│   ├── prob (82, 168, 144, 107) float32
│   └── rel_mass (82, 168, 144, 107) float32
├── L3
│   ├── abc-like (274,) StringDType()
│   ├── descr (274,) StringDType()
│   ├── flags (274,) StringDType()
│   ├── ids (274,) StringDType()
│   ├── mass (274, 168, 144, 107) float32
│   ├── prob (274, 168, 144, 107) float32
│   └── rel_mass (274, 168, 144, 107) float32
├── L4
│   ├── abc-like (716,) StringDType()
│   ├── descr (716,) StringDType()
│   ├── flags (716,) StringDType()
│   ├── ids (716,) StringDType()
│   ├── mass (716, 168, 144, 107) float32
│   ├── prob (716, 168, 144, 107) float32
│   └── rel_mass (716, 168, 144, 107) float32
├── L5
│   ├── abc-like (932,) StringDType()
│   ├── descr (932,) StringDType()
│   ├── flags (932,) StringDType()
│   ├── ids (932,) StringDType()
│   ├── mass (932, 168, 144, 107) float32
│   ├── prob (932, 168, 144, 107) float32
│   └── rel_mass (932, 168, 144, 107) float32
├── bbox
│   ├── ccf_mm
│   │   ├── max (3,) float32
│   │   └── min (3,) float32
│   ├── shape (3,) float32
│   └── voxel_25_um
│       ├── max (3,) float32
│       └── min (3,) float32
├── counts (168, 144, 107) float32
├── evidence (168, 144, 107) float32
├── latent_embedding (45, 168, 144, 107) float32
└── region
    ├── acronyms (176,) StringDType()
    ├── ids (176,) int32
    ├── region_labels (168, 144, 107) uint32
    └── region_masks (176, 168, 144, 107) bool

## Let's process the cell-level spatial object

In [17]:
adata = sc.read_h5ad(SC_ST_IN_FP)

### Let's clean up the pathmat to only include the IDs

In [18]:
pathmat = adata.uns["pathmat"]

# split and keep first token
pathmat_clean = np.vectorize(lambda x: x.split(" ")[0])(pathmat)

adata.uns["pathmat"] = pathmat_clean

In [19]:
adata.uns["pathmat"]

array([['Gb1-2', 'Gb2-39', 'Gb3-124', 'Gb4-319', 'Gb5-428'],
       ['Gb1-2', 'Gb2-39', 'Gb3-124', 'Gb4-320', 'Gb5-429'],
       ['Gb1-2', 'Gb2-39', 'Gb3-124', 'Gb4-321', 'Gb5-430'],
       ...,
       ['NN1-4', 'NN2-14', 'NN3-28', 'NN4-44', 'NN5-44'],
       ['NN1-4', 'NN2-14', 'NN3-28', 'NN4-45', 'NN5-45'],
       ['NN1-4', 'NN2-14', 'NN3-28', 'NN4-46', 'NN5-46']],
      shape=(932, 5), dtype='<U7')

### Let's restructure the membership probabilities, summing up for higher levels

In [20]:
pathmat = adata.uns["pathmat"]  # (n_leaves, 5)
P_L5 = adata.obsm["P_path"]     # (n_cells, n_leaves)

In [ ]:
# Build index maps

level_maps = [] # 4 arrays indicating the cluster label of the path at Level i
level_labels = {} # 4 keys, one for each level, mapping to arrays of unique cluster labels

for lvl in range(4):
    labels = pathmat[:, lvl]
    level_labels[lvl] = np.unique(labels)
    level_maps.append(labels)

In [ ]:

def build_map_matrix(leaves_to_level):
    """
    Construct a sparse mapping matrix from L5 (leaf) clusters to clusters at a
    higher level of the hierarchy.

    Parameters
    ----------
    leaves_to_level : array-like of shape (n_leaves,)
        For each L5 (leaf) cluster, gives the label of its ancestor at one
        hierarchy level.

        Example (mapping leaves to L4):

            leaves_to_level = [
                "L4_A",   # leaf 0 belongs to L4_A
                "L4_A",   # leaf 1 belongs to L4_A
                "L4_B",   # leaf 2 belongs to L4_B
                "L4_B",   # leaf 3 belongs to L4_B
                "L4_C",   # leaf 4 belongs to L4_C
            ]

    Returns
    -------
    M : scipy.sparse.csr_matrix of shape (n_level_nodes, n_leaves)
        Sparse binary mapping matrix.

        Each column corresponds to one L5 leaf.
        Each row corresponds to one unique cluster at the requested level.

        M[i, j] = 1  if leaf j belongs to parent node i
                = 0  otherwise

        Example:

                     leaf0 leaf1 leaf2 leaf3 leaf4
            L4_A       1     1     0     0     0
            L4_B       0     0     1     1     0
            L4_C       0     0     0     0     1

        This matrix is used to aggregate probabilities:

            P_level = P_L5 @ M.T

        where

            P_L5.shape = (n_cells, n_leaves)

        and therefore

            P_level.shape = (n_cells, n_level_nodes)

        Each probability at the higher level is simply the sum of the
        probabilities of all descendant leaves.

    level_names : ndarray of shape (n_level_nodes,)
        The unique cluster IDs corresponding to the rows of M.

        For the example above:

            ["L4_A", "L4_B", "L4_C"]
    """

    # Ensure we are working with a NumPy array.
    leaves_to_level = np.asarray(leaves_to_level)

    # ------------------------------------------------------------------
    # Find all unique parent labels.
    #
    # np.unique(..., return_inverse=True) returns:
    #
    # level_names : the unique labels
    #
    # inv : for every leaf, the integer index of its parent.
    #
    # Example:
    #
    # leaves_to_level = ["A", "A", "B", "B", "C"]
    #
    # level_names = ["A", "B", "C"]
    # inv         = [ 0 ,  0 ,  1 ,  1 ,  2 ]
    #
    # Thus leaf 0 belongs to row 0,
    #      leaf 2 belongs to row 1, etc.
    # ------------------------------------------------------------------
    level_names, inv = np.unique(leaves_to_level, return_inverse=True)

    # Row index of each non-zero entry.
    # One entry per leaf.
    rows = inv

    # Column index is simply the leaf index.
    cols = np.arange(len(leaves_to_level))

    # Every leaf contributes with weight 1 to exactly one parent.
    data = np.ones(len(leaves_to_level), dtype=np.uint8)

    # ------------------------------------------------------------------
    # Construct the sparse matrix.
    #
    # COO format is specified by triplets:
    #
    #     (row, column, value)
    #
    # Example:
    #
    # rows = [0,0,1,1,2]
    # cols = [0,1,2,3,4]
    # data = [1,1,1,1,1]
    #
    # becomes
    #
    #        leaf0 leaf1 leaf2 leaf3 leaf4
    # row0     1     1     0     0     0
    # row1     0     0     1     1     0
    # row2     0     0     0     0     1
    # ------------------------------------------------------------------
    M = sp.coo_matrix(
        (data, (rows, cols)),
        shape=(len(level_names), len(leaves_to_level)),
    ).tocsr()   # CSR is much faster for matrix multiplication.

    return M, level_names

In [23]:
# sum up 

P_levels = {}

for lvl in range(4):
    M, ids = build_map_matrix(level_maps[lvl])
    
    # (cells × leaves) @ (leaves × level_nodes)^T
    # Multiplying the cell × L5 probability matrix with M.T sums the
    # probabilities of all leaves belonging to the same parent cluster.
    P_lvl = P_L5 @ M.T
    
    P_levels[f"L{lvl+1}"] = P_lvl
    adata.uns[f"L{lvl+1}_ID"] = ids
    adata.uns[f"L{lvl+1}_descr"] = [descr_mapper[i] for i in ids]
    adata.uns[f"L{lvl+1}_abc_like"] = [abclike_mapper[i] for i in ids]
    adata.obsm[f"P_L{lvl+1}"] = P_lvl

In [24]:
ids = adata.uns["pathmat"][:, 4]
adata.uns[f"L5_ID"] = ids
adata.uns[f"L5_descr"] = [descr_mapper[i] for i in ids]
adata.uns[f"L5_abc_like"] = [abclike_mapper[i] for i in ids]
adata.obsm[f"P_L5"] = P_L5

### Let's add the max likely descr label and its confidence into .obs

In [25]:
for lvl in range(1, 6):
    P = adata.obsm[f"P_L{lvl}"]

    max_idx = np.asarray(P.argmax(axis=1)).ravel()
    max_prob = np.asarray(P.max(axis=1).toarray()).ravel()

    descr = np.asarray(adata.uns[f"L{lvl}_descr"])
    abc = np.asarray(adata.uns[f"L{lvl}_abc_like"])

    adata.obs[f"L{lvl}_descr_max_p"] = descr[max_idx]
    adata.obs[f"L{lvl}_abc_like_max_p"] = abc[max_idx]
    adata.obs[f"L{lvl}_max_p"] = max_prob

### Let's add the flags

In [ ]:
adata.uns["flags"] = flags_df.reset_index()

In [27]:
adata.uns["flags"].head()

,ids,flag_category
0,Gb1-2,None
1,Gl1-1,None
2,NN1-1,None
3,NN1-2,None
4,NN1-3,None


### Let's tidy up and save

In [28]:
del adata.obsm["P_path"] # redundant as we have L5

In [29]:
adata

AnnData object with n_obs × n_vars = 1401313 × 56939
    obs: 'Dataset', 'Sample', 'Author', 'Technology', 'Donor', 'Sex', 'RegionID', 'RegionAcronym', 'RegionName', 'total_counts', 'total_hvg_counts', 'total_genes', 'IsHY', 'L1_descr_max_p', 'L1_abc_like_max_p', 'L1_max_p', 'L2_descr_max_p', 'L2_abc_like_max_p', 'L2_max_p', 'L3_descr_max_p', 'L3_abc_like_max_p', 'L3_max_p', 'L4_descr_max_p', 'L4_abc_like_max_p', 'L4_max_p', 'L5_descr_max_p', 'L5_abc_like_max_p', 'L5_max_p'
    var: 'gene_stable_ids', 'IsHVG'
    uns: 'pathmat', 'L1_ID', 'L1_descr', 'L1_abc_like', 'L2_ID', 'L2_descr', 'L2_abc_like', 'L3_ID', 'L3_descr', 'L3_abc_like', 'L4_ID', 'L4_descr', 'L4_abc_like', 'L5_ID', 'L5_descr', 'L5_abc_like', 'flags'
    obsm: 'X_embed', 'X_umap', 'X_umap_sc', 'coords_ccf_mm', 'coords_ccf_mm_wrap', 'coords_source', 'coords_voxels_25um', 'coords_voxels_25um_bounded', 'P_L1', 'P_L2', 'P_L3', 'P_L4', 'P_L5'
    varm: 'IsCaptured'

In [31]:
with tracker.register(SC_ST_OUT_FP) as tf:
    tf.set_description('Hypomap3D cell-level spatial object.')
    write_clean_h5ad(adata, tf.filepath)